# Step 6 通時的 word2vec — 時代スライスと Procrustes アラインメント

> **用語の予習・復習**: `docs/glossary.md` の Step 6 を参照。キーワードを見て自分で説明してみてから読むこと。

## このステップの到達目標

1. 独立に学習した word embeddings が**そのままでは比較できない**理由を説明できる
2. 直交 Procrustes 変換によるアラインメントを実装・実行できる
3. 意味変化の指標を計算し，交絡（語彙量・作家）を統制できる
4. 検出された「変化」が本物かを検証する手順を持つ

## 導入：なぜそのまま比べられないのか

word2vec の目的関数は**回転に対して不変**である。
すべてのベクトルを同じ直交行列で回しても，内積（＝コサイン類似度）は変わらない。
したがって学習のたびに空間全体の向きが変わる。

> 明治期モデルの「恋」と昭和期モデルの「恋」のコサイン類似度を直接計算しても，
> **それは意味の違いではなく，座標系の違いを測っている。**

解決は Hamilton, Leskovec & Jurafsky (2016) の方法である。

1. 両モデルに共通する語彙を取る
2. その部分行列どうしを最もよく重ねる**直交行列 R** を求める
   （$R = UV^\top$ where $U\Sigma V^\top = \mathrm{SVD}(B^\top A)$）
3. 回転だけを許すので，**語どうしの距離構造は保たれる**

## 参考
- Hamilton, Leskovec & Jurafsky (2016) Diachronic word embeddings reveal statistical laws of semantic change. *ACL*.
- Kim et al. (2014) Temporal analysis of language through neural language models. *ACL Workshop*.
- Dubossarsky et al. (2017) Outta control: laws of semantic change and inherent biases. *EMNLP*.
- Tabata, T. (2026) Using word embeddings as a semantic approach to key word analysis. *PALA 2026: The Philosophy of Stylistics*, Uppsala, 19–22 August 2026. 資料: <https://tinyurl.com/tabata-pala2026>


In [ ]:
# ---- 共通の準備（毎回このセルから実行する）----------------------------
import os, sys, csv, json, math, random, shutil, subprocess, warnings
import importlib.util
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore', category=FutureWarning)

# リポジトリのルートを自動で探す（my_work/notebooks/ でも notebooks/ でも，上へたどる）
ROOT = Path.cwd()
while not (ROOT / 'config' / 'pipeline.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
print('ROOT =', ROOT)
if Path.cwd().resolve() == (ROOT / 'notebooks').resolve():
    print('[注意] 配布版（notebooks/）を直接開いている。実行すると次の git pull が止まる。\n'
          '       python scripts/copy_notebooks.py でコピーを作り，my_work/notebooks/ の方を開くこと。')

# 日本語フォント（□ にならないように）
for cand in ['Hiragino Sans', 'Yu Gothic', 'Meiryo',
             'Noto Sans CJK JP', 'IPAexGothic', 'MS Gothic']:
    if cand in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams['font.family'] = cand
        break
else:
    print('[!] 日本語フォントが見つかりません。docs/00_setup_students.md §1.7 を参照。')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ---- 図はすべて SVG（ベクタ）で保存する ---------------------------------
# 論文・スライドに載せる図は拡大しても劣化してはならない。PNG は解像度が
# 固定されるので，投影や印刷で文字が潰れる。SVG なら任意の倍率で鮮明で，
# Illustrator / Inkscape で軸ラベルだけを直すこともできる。
FIG_EXT   = 'svg'
RASTER_DPI = 200          # rasterized=True の要素にだけ効く
plt.rcParams['svg.fonttype']       = 'path'   # 文字をアウトライン化して環境非依存に
plt.rcParams['savefig.transparent'] = False
# 画面へのインライン表示は既定（PNG）のままにする。
# InlineBackend.figure_formats を 'svg' に変えると，JupyterLab や
# VS Code の版によっては図がまったく表示されなくなることがある。
# **保存されるファイルは SVG** なので，論文・スライドに使うほうは
# ベクタで手元に残る。画面で拡大して見たいときは save_fig が表示する
# パスの .svg をブラウザで開くこと。

def need(path, hint=''):
    """必要な入力があるか確かめる。無ければ**理由を表示して** False を返す。

    セルを `if p.exists():` で囲むと，入力が無いときに何も起きない。
    受講生には「壊れている」と「まだ前の工程を走らせていない」の区別が
    つかず，図が出ないという相談の大半がこれである。必ず理由を出す。
    """
    p = Path(path)
    try:
        ok = p.is_file() or (p.is_dir() and any(p.iterdir()))
    except OSError:
        ok = False
    if not ok:
        print(f'[未実行] {p} がありません。')
        if hint:
            print(f'         {hint}')
        print('         この Step の前のセルを上から順に実行すること。'
              '\n         それでも出ない場合は，前の Step のノートブックが'
              '最後まで通っているか確認する。')
    return ok


# 旧名 → 新名。**中身は 0–1 の割合なので per cent は誤称**である。
# 旧名の列を持つ古い出力も読めるように，読み替えを残す。
LEGACY_COLS = {'df_all_pct': 'df_all_prop', 'df_in_pct': 'df_in_prop'}


def read_table(path, **kw):
    """CSV を読み，**古い列名があれば新しい名前に読み替える**。

    列名の約束：割合（0–1）は ``_prop`` / ``_ratio`` / ``_share``，
    百分率（0–100）だけを ``_pct`` と綴る。``df_all_prop`` が 0.1584 なら
    15.84 % の意である。読み替えたときは黙らずに知らせる — 黙って直すと，
    手元の CSV と教材の列名が食い違っていることに気づけないため。
    """
    d = pd.read_csv(path, **kw)
    old = {k: v for k, v in LEGACY_COLS.items()
           if k in d.columns and v not in d.columns}
    if old:
        d = d.rename(columns=old)
        print('[note] 古い列名を読み替えた: '
              + '，'.join(f'{k}→{v}' for k, v in old.items())
              + '\n       07_descriptive_stats.py を走らせ直すと'
                '新しい名前で書き出される。')
    return d


def load_meta(path=None, analysis_only=True):
    """メタデータを読む。既定では**分析に使う行だけ**を返す。

    落とすのは2種類。書誌としては残すが，集計に足してはいけない行である。
      superseded … v1 の合本。増補で分冊ごとに取り直したので，足すと
                   同じ作品を二重に数える
      merged     … 分冊。03b で canonical の巻に本文を統合したので，
                   この行はもう本文を持たない（『夜明け前』『家』）
      too_short  … 1チャンクにも満たず，チャンク単位の分析に乗らない

    生の表がほしいときは ``analysis_only=False``。
    """
    df = pd.read_csv(path or META)
    if analysis_only and 'completeness' in df.columns:
        drop = df['completeness'].isin(['superseded', 'merged', 'too_short'])
        if drop.any():
            names = '，'.join(df.loc[drop, 'title_aozora'].astype(str))
            print(f'[meta] 分析から除外 {int(drop.sum())} 行: {names}')
        df = df[~drop].reset_index(drop=True)
    return df


def w_ljust(text, width):
    """全角を2桁と数えて左詰めする。

    ``f'{s:<26}'`` は**文字数**で詰めるので，日本語の作品名を並べると
    桁が揃わない（全角は2桁ぶんの幅を占める）。表として読ませるなら
    表示幅で詰めること。

    **なお，一覧を出すなら ``show()`` で表にするほうがよい**（下記）。
    この関数は，表にしにくいもの（KWIC の前後文脈など）を print で
    並べるときに使う。
    """
    import unicodedata
    text = str(text)
    w = sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in text)
    return text + ' ' * max(0, width - w)


# ---------------------------------------------------------------------------
# 分析結果の表示
# ---------------------------------------------------------------------------
# **一覧は print ではなく表で出す。**
#   * print は桁が揃わない（全角の幅）。数字の比較がしにくい
#   * 列に名前が付かないので，あとで見返したときに何の数字か分からない
#   * 並べ替えも絞り込みもできない
# 表にすると，列名がそのまま「何を測ったか」の記録になる。
# **ただし何でも表にするのではない。** 単発の数値・警告・KWIC の前後文脈は
# 文のほうが読みやすい。目安は「2列以上あるか」「行が並ぶか」。
TABLE_STYLES = [
    {'selector': 'caption',
     'props': [('caption-side', 'top'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '0 0 .4em 0'),
               ('color', '#33322e'), ('font-size', '95%')]},
    {'selector': 'th',
     'props': [('background', '#f2f2ef'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '.26em .7em'),
               ('border-bottom', '1px solid #c6c5bd'), ('white-space', 'nowrap')]},
    {'selector': 'td',
     'props': [('padding', '.22em .7em'), ('border-bottom', '1px solid #ecebe6')]},
    {'selector': 'tbody tr:hover td', 'props': [('background', '#f7f7f4')]},
]


def show(df, caption='', fmt=None, index=False, header=True, na='—', align=None):
    """DataFrame を表として表示する。Jupyter 以外でも落ちない。

    ``fmt`` は pandas の ``Styler.format`` に渡す辞書
    （例 ``{'一致率': '{:.1%}', 'G²': '{:.0f}'}``）。
    数値の列は自動で右寄せにする。``align`` で列ごとに寄せを指定できる。
    KWIC の左文脈を ``align={'左文脈': 'right'}`` にすると，
    **キーワードが縦に揃う**（等幅フォントに頼らずに揃う）。
    返り値は ``df`` なので ``t = show(df)`` として続けて使える。
    """
    if isinstance(df, pd.Series):
        df = df.to_frame()
    try:
        from IPython.display import display as _display
        st = df.style.format(fmt, na_rep=na) if fmt else df.style.format(na_rep=na)
        st = st.set_table_styles(TABLE_STYLES)
        num = list(df.select_dtypes('number').columns)
        if num:
            st = st.set_properties(subset=num, **{'text-align': 'right'})
        for col, side in (align or {}).items():
            if col in df.columns:
                st = st.set_properties(subset=[col],
                                       **{'text-align': side,
                                          'white-space': 'pre'})
        if caption:
            st = st.set_caption(caption)
        if not index:
            st = st.hide(axis='index')
        if not header:
            st = st.hide(axis='columns')
        _display(st)
    except Exception:                                   # noqa: BLE001
        # ノートブックの外（スクリプトから import したとき）でも読める形
        if caption:
            print(caption)
        print(df.to_string(index=index, header=header))
    return df


def grid(items, ncol=8, caption=''):
    """語の並びを ``ncol`` 列の表にして表示する。

    40 語を1行に流すと折り返しで読めない。列に切ると目で追える。
    順位が要るなら ``show()`` に順位列を付けた表を渡すこと。
    """
    items = [str(x) for x in items]
    rows = [items[i:i + ncol] for i in range(0, len(items), ncol)]
    rows = [r + [''] * (ncol - len(r)) for r in rows]
    t = pd.DataFrame(rows, columns=[f'_{i}' for i in range(ncol)])
    return show(t, caption=caption, header=False)


def work_rows(meta_df=None):
    """``work_stem`` からメタデータの行を引く辞書を作る。

    ``meta_df`` を省くと**分析対象外の行も含めた全件**から作る。
    表示用の名前は，分析から外した作品についても引けるほうがよい。

    **キーの綴りに注意。** 青空文庫の作品 ID は索引では 0 埋めされていない
    （``1743``）が，本パイプラインのファイル名は6桁に 0 埋めしてある
    （``000119_001743``）。素朴に連結すると ``000119_1743`` となり，
    **1件も一致しない**。辞書は空振りしても例外を出さないので，
    誰の何だか分からないまま最後まで通ってしまう。両方の綴りを登録する。

    ``file_v1`` は増補した作品では空である。``os.path.splitext(nan)`` は
    例外になるので，文字列であることを確かめてから使う。
    """
    if meta_df is None:
        meta_df = load_meta(analysis_only=False)
    d = {}
    for _, r in meta_df.iterrows():
        fv = r.get('file_v1')
        if isinstance(fv, str) and fv.strip():
            d[os.path.splitext(fv)[0]] = r
        pid = str(r.get('aozora_person_id') or '').strip()
        wid = str(r.get('aozora_work_id') or '').strip()
        if pid and wid and pid.lower() != 'nan' and wid.lower() != 'nan':
            for k in (f'{pid.zfill(6)}_{wid.zfill(6)}',
                      f'{pid}_{wid}', f'{pid.zfill(6)}_{wid}'):
                d[k] = r
    return d


def work_labels(meta_df=None, maxlen=12, with_year=False):
    """``work_stem`` → ``作者『作品』`` の対応表を返す。

    ``000119_001743`` と出されても誰の何だか分からない。距離の近い
    ペアを見るときに**どの作家のどの作品か**が分からなければ，
    「作家効果か時代効果か」という問いにそもそも答えられない。
    表示するときは必ずこれを通すこと。
    """
    out = {}
    for k, r in work_rows(meta_df).items():
        t = str(r.get('title_aozora') or '')
        lab = f"{r.get('author_ja', '?')}『{t[:maxlen]}』"
        if with_year and str(r.get('year_first') or '').strip():
            lab += f"({r['year_first']})"
        out[k] = lab
    return out


def attach_meta(df, cols, stem_col='work_stem', meta_df=None, quiet=False,
                fill_blank=True):
    """``df`` に足りないメタデータの列を，``work_stem`` から引いて補う。

    ``fill_blank=True``（既定）なら，**列はあるのに値が空**のセルも補う。
    列が無いより，列があって半分が空のほうが危ない。列が無ければ
    ``AttributeError`` で止まるが，値が空だと**図がそのまま描けてしまう**。
    たとえば突合が外れて多くの作品の ``year_first`` が空になっても，図は
    描けてしまい，それらが「初出年不明」の灰色で並ぶだけである。
    値の空きも数えて報告し，ここで補えるものは補う。

    分析スクリプトの出力は，その分析に要る列しか書かない。
    ``09_doc2vec.py`` の ``work_vectors.csv`` に ``genre_main`` が無いのは
    その一例である。ノートブックで ``wv.genre_main`` と書けば
    ``AttributeError: 'DataFrame' object has no attribute 'genre_main'``
    になるが，**足りないのは列であって情報ではない**。
    メタデータ表には必ずあるのだから，ここで引いて補えばよい。

    出力 CSV の列構成に図の描画が依存するのは弱い。分析スクリプトを
    書き換えるたびに図が落ちる。図の側で「要る列を宣言して取りに行く」
    ほうが，どちらを先に走らせても通る。

    引けなかった列は空のまま残し ``[warn]`` を出す。図が落ちるより，
    「この軸は色分けできなかった」と分かったうえで出るほうがよい。
    """
    df = df.copy()
    if stem_col not in df.columns:
        if not quiet:
            print(f'[warn] {stem_col} 列が無いので補完できない: {list(cols)}')
        for c in cols:
            if c not in df.columns:
                df[c] = ''
        return df

    rows = work_rows(meta_df)

    # **まずキーが合っているかを見る。** 合っていなければ何も補えない。
    # 「1件も合わない」のはたいてい 0 埋めの綴り違いで，黙って通すと
    # 全部の軸が空のまま図になる。
    stems = df[stem_col].astype(str)
    found = stems.map(lambda s: s in rows)
    if not quiet and not found.all():
        n_miss = int((~found).sum())
        lv = 'FATAL' if found.sum() == 0 else 'warn '
        print(f'[{lv}] {stem_col} がメタデータと突合できない行が '
              f'{n_miss}/{len(df)} 件ある: '
              + '，'.join(stems[~found].head(4)))
        if found.sum() == 0:
            print('        **1件も合っていない。** 作品 ID の 0 埋めの'
                  '綴り違いを疑うこと（例 000119_1743 と 000119_001743）。')
            print('        この表を作ったスクリプトのキーの作り方を直すこと。')

    def _blank(v):
        return v is None or str(v).strip().lower() in ('', 'nan', 'none')

    for c in cols:
        if c not in df.columns:
            vals = [(lambda r: '' if r is None or _blank(r.get(c))
                     else r.get(c))(rows.get(s)) for s in stems]
            df[c] = vals
            n = int(sum(1 for v in vals if str(v).strip()))
            if not quiet:
                mark = 'ok  ' if n == len(df) else 'warn'
                print(f'[{mark}] {c} をメタデータから補完: {n}/{len(df)} 件')
            continue

        if not fill_blank:
            continue
        # 列はある。空のセルだけを埋める。
        blank = df[c].map(_blank)
        if not blank.any():
            continue
        filled = 0
        vals = df[c].tolist()
        for i, (s, is_blank) in enumerate(zip(stems, blank)):
            if not is_blank:
                continue
            r = rows.get(s)
            if r is not None and not _blank(r.get(c)):
                vals[i] = r.get(c)
                filled += 1
        df[c] = vals
        if not quiet:
            left = int(sum(1 for v in vals if _blank(v)))
            mark = 'fix ' if left == 0 else 'warn'
            print(f'[{mark}] {c} は {int(blank.sum())}/{len(df)} 件が空だった'
                  f' → {filled} 件をメタデータから補完'
                  + ('' if left == 0 else f'（なお {left} 件が空）'))
            if left:
                print('        **その列で色分けする図・集計は，この件数を'
                      '報告に書くこと。**')
    return df


def label_points(ax, xs, ys, texts, fontsize=8, color='#333333', pad=4,
                 leader='line', leader_min=13, crowd_r=26,
                 leader_color='#8a8a83'):
    """散布図の注記を，重ならない位置だけに置き，遠いものは引き出し線で結ぶ。

    素朴に ``ax.annotate(t, (x, y))`` と書くと，**注目すべき点ほど一箇所に
    固まる**ので注記が必ず重なって読めなくなる。文語標識の上位は
    どれも口語標識がほぼ 0 で，対数軸の右下隅に密集するのが典型である。

    そこで点の周囲を順に試し，他の注記とも他の点とも重ならず，かつ軸の
    内側に収まる位置があればそこに置く。どこにも置けない注記は**置かずに
    数だけ報告する**。読めない字を重ねるより，図の外（下の表）で番号から
    引くほうがよい。

    **離れた位置に置いた注記は，引き出し線で点と結ぶ。** 避けた結果として
    注記は点から離れるので，線が無いとどの点の名前なのか分からなくなる
    ——密集した領域では隣の点の名前だと読まれる。線があれば，遠くへ逃がす
    ことに副作用が無くなるので，**近くに空きが無い注記も置ける**ようになる
    （候補の輪を 24・30 ポイントまで広げてあるのはそのため）。

    ``leader``
        ``'line'``（既定）… 矢じりの無い細線で結ぶ。図版の慣例はこちら。
        6.5pt の文字に矢じりを付けると，マーカーそのものを覆って点が読めなくなる
        ``'arrow'`` … 小さな矢じりを付ける
        ``'none'`` … 結ばない（従来どおり）
    ``leader_min``
        この距離（ポイント）より遠くに置いた注記を結ぶ。既定は 0，
        つまり**すべて結ぶ**。注記は必ず点から離れた位置に置かれるので，
        離れている以上「どの点の名前か」は線でしか確定しない。
        線を省くと，隣の点の名前だと読まれる余地が残る。
    ``crowd_r``
        注記の近くに**自分以外の点**がこの半径（ピクセル）内にあるかを
        見る。``leader_min`` を上げて線を減らしたときでも，
        紛れる相手が居る注記だけは必ず結ぶための保険である。

    表示座標で矩形の重なりを見るので，**軸の位置が確定してから**呼ぶ。
    ``fig.tight_layout()`` はこの関数より**前**に呼ぶこと（後で呼ぶと軸が
    動き，せっかく避けた位置がずれる）。戻り値は置けた注記の数。
    """
    from matplotlib.transforms import Bbox
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    texts = list(texts)
    if len(texts) == 0:
        return 0
    # **長さが違えば黙って切り詰めずに止める。** zip は短いほうに合わせるので，
    # 座標だけを絞り込んで名前を絞り忘れると，先頭から順に**別の作品の名前**が
    # 貼られた図が，何の警告も出さずに出来上がる。これがいちばん重い事故である。
    if not (len(xs) == len(ys) == len(texts)):
        raise ValueError(
            f'label_points: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'名前={len(texts)}）。座標と名前を同じ添字で絞り込むこと。'
            'たとえば P[pick,0] と組むのは names[pick] であって names ではない。')
    fig = ax.figure
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    axbb = ax.get_window_extent(renderer=ren)

    def pad_box(b, w=2.0, h=1.5):
        """注記の矩形に**絶対量の余白**を足す。

        倍率（``expanded(1.08, …)``）では足りない。1桁の数字は幅 8px ほど
        なので 8% は 0.6px にしかならず，隣り合う注記が触れるほど近くても
        「重なっていない」と判定される。**``21`` が「21」と読める**のは
        これが原因である。文字の大小によらず一定の余白を確保する。
        """
        return Bbox.from_extents(b.x0 - w, b.y0 - h, b.x1 + w, b.y1 + h)

    def bb_of(ann):
        # Annotation 自身の get_window_extent を使うこと。
        # Text.get_window_extent(ann, ...) を呼ぶと xy の位置が無視され，
        # xytext のオフセットを絶対座標と見た矩形が返って判定が壊れる。
        return ann.get_window_extent(renderer=ren)

    blocked = []
    for coll in ax.collections:
        try:
            for p in coll.get_offsets():
                px, py = ax.transData.transform(p)
                blocked.append(Bbox.from_bounds(px - pad, py - pad,
                                                2 * pad, 2 * pad))
        except Exception:                                    # noqa: BLE001
            pass

    # **まっすぐ真上・真下を先に試す。** 点の直上に中央揃えで置ければ，
    # それがいちばん素直で，引き出し線も要らない。横へずらすのは，
    # 直上が塞がっていたときの次善である。
    #
    # 横へずらす輪は 12 ポイントから始める。**線が線として見える長さを
    # 確保する**ため。8 ポイントに置くと引き出し線が3ピクセルの点にしか
    # ならず，汚れと区別がつかない。
    # **真上に置けなければ，まず真上へ逃がす。** 横へ逃がすと注記の左右の
    # 順序が点の順序と入れ替わり，引き出し線も交差する。真上に段を重ねる
    # 限り，x は動かないので順序は必ず保たれる。横へずらすのは最後。
    # **横のずらし幅は小さく取る。** 横へ 30 ポイントも動かすと，注記が
    # 隣の点の真上に乗り，引き出し線で結んでも読みにくい。真上に段を
    # 重ねるほうが先で（x が動かないので順序が保たれる），横は 8→18
    # ポイントの範囲に収める。
    CAND = [(0, 9), (0, -11), (0, 20), (0, -22), (0, 31), (0, -33),
            (8, 5), (-8, 5), (8, -12), (-8, -12),
            (11, 0), (-11, 0),
            (13, 9), (-13, 9), (13, -16), (-13, -16),
            (18, 0), (-18, 0), (18, 14), (-18, 14),
            (0, 42), (0, -44)]

    def _ha(dx):
        # dx が 0 なら**中央揃え**。ここを 'left' にすると，真上に置いた
        # つもりの注記が文字幅の半分だけ右にずれ，隣の点の上に乗る。
        return 'center' if dx == 0 else ('left' if dx > 0 else 'right')
    placed, chosen, skipped = [], [], 0
    for x, y, t in zip(xs, ys, texts):
        # **自分が指している点は避けない。** 除かないと，注記は必ず
        # 自分の点の隣に来るので全部「重なる」と判定され，1つも置けない。
        ox, oy = ax.transData.transform((x, y))
        near = [b for b in blocked
                if not (abs((b.x0 + b.x1) / 2 - ox) < 1
                        and abs((b.y0 + b.y1) / 2 - oy) < 1)]
        for dx, dy in CAND:
            ann = ax.annotate(str(t), (x, y), textcoords='offset points',
                              xytext=(dx, dy), fontsize=fontsize, color=color,
                              ha=_ha(dx),
                              va='bottom' if dy >= 0 else 'top', zorder=6)
            bb = pad_box(bb_of(ann))
            inside = (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                      and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1)
            if inside and not any(bb.overlaps(b) for b in placed + near):
                placed.append(bb)
                chosen.append((ann, x, y, str(t), dx, dy))
                break
            ann.remove()
        else:
            skipped += 1

    # ---- 交差をほどく ----------------------------------------------------
    # **引き出し線が交差すると，注記の左右の順序が点の順序と入れ替わる。**
    # 文語標識の上位のように順位そのものが意味を持つ図では，2 と 3 が
    # 入れ替わって並ぶだけで読み違えられる。交差している2件を見つけ，
    # **位置を入れ替えて交差が解ければ入れ替える**（2-opt）。
    def _cross(p, q, r, s):
        def o(a, b, c):
            return ((b[0] - a[0]) * (c[1] - a[1])
                    - (b[1] - a[1]) * (c[0] - a[0]))
        return (((o(r, s, p) > 0) != (o(r, s, q) > 0))
                and ((o(p, q, r) > 0) != (o(p, q, s) > 0)))

    kpt = fig.dpi / 72.0

    def _seg(i):
        ann, x, y, t, dx, dy = chosen[i]
        ox, oy = ax.transData.transform((x, y))
        return (ox, oy), (ox + dx * kpt, oy + dy * kpt)

    def _set_off(i, dx, dy):
        ann, x, y, t, _, _ = chosen[i]
        ann.set_position((dx, dy))
        ann.set_ha(_ha(dx))
        ann.set_va('bottom' if dy >= 0 else 'top')
        chosen[i] = (ann, x, y, t, dx, dy)

    def _fits(i, bb):
        ox, oy = ax.transData.transform((chosen[i][1], chosen[i][2]))
        if not (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1):
            return False
        return not any(bb.overlaps(b) for b in blocked
                       if abs((b.x0 + b.x1) / 2 - ox) > 1
                       or abs((b.y0 + b.y1) / 2 - oy) > 1)

    swaps = 0
    for _ in range(3):
        improved = False
        for i in range(len(chosen)):
            for j in range(i + 1, len(chosen)):
                if not _cross(*_seg(i), *_seg(j)):
                    continue
                di, dj = chosen[i][4:6], chosen[j][4:6]
                _set_off(i, *dj)
                _set_off(j, *di)
                bi = pad_box(bb_of(chosen[i][0]))
                bj = pad_box(bb_of(chosen[j][0]))
                others = [b for k2, b in enumerate(placed) if k2 not in (i, j)]
                good = (not bi.overlaps(bj)
                        and not any(bi.overlaps(b) or bj.overlaps(b)
                                    for b in others)
                        and _fits(i, bi) and _fits(j, bj)
                        and not _cross(*_seg(i), *_seg(j)))
                if good:
                    placed[i], placed[j] = bi, bj
                    swaps += 1
                    improved = True
                    continue
                _set_off(i, *di)
                _set_off(j, *dj)

                # 入れ替えが収まらないときは，**片方を別の候補位置へ動かす**。
                # 入れ替えは2つの箱の大きさが違うと失敗しやすい（数字1桁と
                # 作者名では幅が違う）。動かすほうは箱の大きさが変わらない。
                moved = False
                for who, other in ((i, j), (j, i)):
                    d0 = chosen[who][4:6]
                    for cx, cy in CAND:
                        if (cx, cy) == tuple(d0):
                            continue
                        _set_off(who, cx, cy)
                        bw = pad_box(bb_of(chosen[who][0]))
                        rest = [b for k2, b in enumerate(placed) if k2 != who]
                        if (_fits(who, bw)
                                and not any(bw.overlaps(b) for b in rest)
                                and not _cross(*_seg(who), *_seg(other))
                                and not any(_cross(*_seg(who), *_seg(k2))
                                            for k2 in range(len(chosen))
                                            if k2 != who)):
                            placed[who] = bw
                            swaps += 1
                            moved = improved = True
                            break
                        _set_off(who, *d0)
                    if moved:
                        break
        if not improved:
            break

    # ---- 引き出し線 ------------------------------------------------------
    # **線は配置が全部決まってから付ける。** arrowprops を付けた
    # Annotation の get_window_extent は「文字＋線」の外接矩形を返すので，
    # 配置の判定に使うと自分の点と必ず重なり，1件も置けなくなる。
    n_leader = 0
    if leader in ('line', 'arrow'):
        style = '-' if leader == 'line' else '-|>'
        for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
            ox, oy = ax.transData.transform((x, y))

            def dist_to_box(px, py, b=bb):
                # 文字の矩形から点までの距離。矩形の中なら 0。
                ddx = max(b.x0 - px, 0, px - b.x1)
                ddy = max(b.y0 - py, 0, py - b.y1)
                return (ddx * ddx + ddy * ddy) ** .5

            # **素直に置けたものには線を引かない。** 点の直上（または直下）に
            # 中央揃えで載っていて，しかもその注記にいちばん近い点が自分の
            # 点であれば，どの点の名前かは見れば分かる。線はかえって邪魔
            # である。横へ逃がしたものだけを結ぶ。
            if dx == 0 and abs(dy) <= 12:
                continue            # 点の真上・真下の一段目 → 線は要らない
            # それ以外は結ぶ。**段を上げたものも結ぶ。** 一段上げた注記の
            # 真下には別の点の注記が入るので，どちらの点のものか分からなく
            # なる。横へずらしたものは言うまでもない。
            d_other = min(
                (dist_to_box((b.x0 + b.x1) / 2, (b.y0 + b.y1) / 2)
                 for b in blocked
                 if abs((b.x0 + b.x1) / 2 - ox) > 1
                 or abs((b.y0 + b.y1) / 2 - oy) > 1),
                default=float('inf'))
            if (dx * dx + dy * dy) ** .5 < leader_min and d_other >= crowd_r:
                continue
            ann.remove()
            ax.annotate(t, (x, y), textcoords='offset points',
                        xytext=(dx, dy), fontsize=fontsize, color=color,
                        ha=_ha(dx),
                        va='bottom' if dy >= 0 else 'top', zorder=6,
                        arrowprops=dict(arrowstyle=style, linewidth=.55,
                                        color=leader_color, alpha=.9,
                                        shrinkA=1.5, shrinkB=2.5,
                                        mutation_scale=7))
            n_leader += 1

    # ---- 誤読の自己点検 --------------------------------------------------
    # **注記の最寄りの点が自分の点でないものを数える。** これが
    # 「ラベルとデータ点がずれて見える」の正体である。引き出し線を
    # 引いてあれば誤読にはならないが，線を切った設定では危険なので，
    # そのときだけ警告を出す。
    risky = []
    for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
        ox, oy = ax.transData.transform((x, y))
        cx, cy = (bb.x0 + bb.x1) / 2, (bb.y0 + bb.y1) / 2
        d_own = ((cx - ox) ** 2 + (cy - oy) ** 2) ** .5
        d_other = min(
            (((b.x0 + b.x1) / 2 - cx) ** 2 + ((b.y0 + b.y1) / 2 - cy) ** 2) ** .5
            for b in blocked
            if abs((b.x0 + b.x1) / 2 - ox) > 1 or abs((b.y0 + b.y1) / 2 - oy) > 1
        ) if len(blocked) > 1 else float('inf')
        if d_other < d_own:
            risky.append(t)
    left = sum(1 for i in range(len(chosen)) for j in range(i + 1, len(chosen))
               if _cross(*_seg(i), *_seg(j)))
    if skipped:
        print(f'[fig] 重なるため {skipped} 件の注記を省いた（表で引くこと）')
    if left:
        print(f'[warn] 引き出し線の交差が {left} 件ほどけなかった。'
              '注記の左右の順序が点の順序と食い違う。'
              '注記を短くするか，件数を減らすこと。')
    if risky:
        head = '，'.join(str(r) for r in risky[:6])
        more = f' ほか{len(risky) - 6}件' if len(risky) > 6 else ''
        if leader in ('line', 'arrow'):
            print(f'[fig] {len(risky)} 件の注記は別の点のほうが近い'
                  f'（{head}{more}）。引き出し線で結んであるので読み違えない。')
        else:
            print(f'[warn] {len(risky)} 件の注記は**別の点のほうが近い**'
                  f'（{head}{more}）。leader="none" では読み違えが起きる。')
    return len(placed)
def _proj_versions():
    """射影に関わる版を並べる（うまくいかないときの手がかり）。"""
    import importlib
    out = []
    for nm in ['numpy', 'numba', 'llvmlite', 'pynndescent', 'sklearn']:
        try:
            out.append(f'{nm} ' + str(getattr(importlib.import_module(nm),
                                              '__version__', '?')))
        except Exception:                                    # noqa: BLE001
            out.append(f'{nm} ×')
    return '／'.join(out)

def umap_diagnosis(e):
    """UMAP が使えないときに，**何をすればよいか**を出す。"""
    import sys
    print(f'[NG  ] UMAP が使えない: {type(e).__name__}: {e}')
    print(f'       このカーネルの Python = {sys.executable}')
    print(f'       {_proj_versions()}')
    if isinstance(e, ModuleNotFoundError):
        # **入れた先とカーネルの環境が違う**のが圧倒的に多い。
        # uv add は「プロジェクト（pyproject.toml のある場所）」単位なので，
        # dh_project/pyproject.toml が無い，または dh_project の外に clone
        # した場合は，uv は別のプロジェクトに入れる。カーネルの .venv には入らない。
        print('       **この環境には入っていない。** 入れた先が違う可能性が高い')
        print('       （uv add はプロジェクト単位。~/Documents/dh_project に')
        print('        pyproject.toml が無いと，別のプロジェクトに入る）。')
        print('       この環境を名指しして入れるのが確実:')
        import platform as _pf
        if sys.platform == 'darwin' and _pf.machine() == 'x86_64':
            # **Intel Mac は版を固定する。** llvmlite の x86_64 wheel は
            # 0.45.1 が最後で，0.46 以降は arm64 のみ。固定しないと
            # ソースからのビルドに落ち，Homebrew の LLVM と版が合わずに
            # 失敗する（llvmlite 0.49 は LLVM 22 を要求）。
            print('       （Intel Mac なので**版を固定する**。'
                  'llvmlite の x86_64 wheel は 0.45.1 が最後）')
            print(f'         uv pip install --python "{sys.executable}" \\')
            print('             --only-binary :all: \\')
            print('             "numba==0.62.1" "llvmlite==0.45.1" '
                  '"numpy<2.4" umap-learn')
        else:
            print(f'         uv pip install --python "{sys.executable}" umap-learn')
        print('       入れたら**カーネルを再起動**して，このセルから実行し直す。')
    else:
        print('       import は通るが使えない型の失敗である'
              '（別パッケージの umap／numba と numpy の版違い／'
              'numba のキャッシュ）。')
    print('       切り分けの全項目:')
    print('         import sys, subprocess; print(subprocess.run('
          '[sys.executable,')
    print("             str(ROOT/'scripts'/'check_umap.py')], "
          'capture_output=True,')
    print('             text=True).stdout)')

def project(Xn, how='umap', seed=20260920, n_neighbors=15, min_dist=0.12,
            perplexity=30):
    """高次元の行列を2次元に落とす。**どの方法で落としたかを必ず返す。**

    ``how`` は ``'umap'``／``'tsne'``／``'auto'``。既定の ``'umap'`` は，
    使えなければ**止まって理由を出す**。``'auto'`` のときだけ t-SNE に落ちる。
    **黙って別の方法に替えないのが肝心である**（図は出るが塊の見え方は
    変わるので，環境の問題を分析結果と読み違える）。

    Step 5 の §3（主成分分析との比較）と §4（ギャラクシー）が共有する。
    """
    import importlib
    Xn = np.asarray(Xn, dtype=np.float32)
    if how in ('auto', 'umap'):
        try:
            m = importlib.import_module('umap')
            if not hasattr(m, 'UMAP'):
                # PyPI には umap（別物）と umap-learn（本物）がある。
                # pip install umap をしていると import umap はそちらを拾う。
                raise ImportError(
                    f'umap に UMAP クラスが無い（{getattr(m, "__file__", "?")}）。'
                    '別パッケージの umap が入っている。'
                    'umap を外して umap-learn を入れること')
            P = m.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                       metric='cosine', random_state=seed).fit_transform(Xn)
            return (np.asarray(P, dtype=np.float32),
                    f'UMAP {getattr(m, "__version__", "")}'
                    f' (n_neighbors={n_neighbors}, min_dist={min_dist}, cosine)')
        except Exception as e:                               # noqa: BLE001
            umap_diagnosis(e)
            if how == 'umap':
                # **黙って別の方法に替えない。** どうしても t-SNE で
                # 進めたいときは 'tsne' と明示し，報告にもそう書くこと。
                raise
            print('[warn] how="auto" なので t-SNE に切り替える。'
                  '**図と報告に t-SNE と書くこと。**')
    from sklearn.manifold import TSNE
    P = TSNE(n_components=2, perplexity=perplexity, metric='cosine',
             init='pca', random_state=seed).fit_transform(Xn)
    return (np.asarray(P, dtype=np.float32),
            f't-SNE (perplexity={perplexity}, cosine)')


def proj_quality(Xn, P, k=10):
    """射影がどれだけ嘘をついているかを3つの数で返す。

    ``trust``  … 2次元で近く見える点が原空間でも近いか（局所・1が最良）
    ``keep``   … 原空間の上位 k 近傍のうち画面でも上位 k に入る語数
    ``rho``    … 原空間の距離と画面の距離の順位相関（**大域**の保存）

    局所（trust・keep）と大域（rho）は別物である。**UMAP は局所に強く，
    主成分分析は大域に強い**——これを目で見ずに数で確かめるための関数。
    """
    from scipy.spatial.distance import pdist
    from scipy.stats import spearmanr
    from sklearn.manifold import trustworthiness
    Xn, P = np.asarray(Xn, np.float32), np.asarray(P, np.float32)
    S = Xn @ Xn.T
    np.fill_diagonal(S, -np.inf)
    nn_t = np.argsort(-S, axis=1)[:, :k]
    d2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(d2, np.inf)
    nn_p = np.argsort(d2, axis=1)[:, :k]
    keep = np.array([len(set(a) & set(b)) for a, b in zip(nn_t, nn_p)])
    trust = float(trustworthiness(Xn, P, n_neighbors=k, metric='cosine'))
    rho = float(spearmanr(pdist(Xn, 'cosine'), pdist(P))[0])
    return {'trust': trust, 'keep': keep, 'rho': rho}


def reserve_right(fig, frac=0.80):
    """面の外に凡例を置いた図で，**右に余白を確保する**。

    ``tight_layout()`` は面の外に置いた凡例を数えないので，そのままだと
    凡例が図の枠から出る。静止版は ``bbox_inches='tight'`` で救われるが，
    **HTML に埋め込む版は切り取らない**（切り取ると点の位置の割合が
    ずれる）ので，凡例が切れて読めなくなる。

    ``tight_layout()`` の**後**，注記（``label_points``）の**前**に呼ぶ。
    """
    fig.subplots_adjust(right=frac)


def save_fig(fig, stem, out=None):
    """図を SVG で保存してパスを表示する。

    stem は拡張子なしの名前（例 'Step1_period_balance'）。
    点が数千個ある散布図は，散布図だけ rasterized=True にしておくと
    軸と文字はベクタのままファイルが軽くなる。
    """
    d = Path(out) if out else OUT
    d.mkdir(parents=True, exist_ok=True)
    path = d / f'{stem}.{FIG_EXT}'
    fig.savefig(path, format=FIG_EXT, dpi=RASTER_DPI, bbox_inches='tight')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB)')
    return path


# ---------------------------------------------------------------------------
# 対話的な図（SVG はそのまま残す）
# ---------------------------------------------------------------------------
# 散布図の点が何百個あると，注記を付けられるのはごく一部である。残りの点は
# 「どの語か」が分からないまま眺めることになる。かといって全点に名前を
# 付ければ図は読めない。
#
# そこで**同じ図から2つ出す**。
#   * ``<stem>.svg``  … 論文・配布用。加筆も拡大も自由
#   * ``<stem>.html`` … 授業・探索用。SVG をそのまま埋め込み，
#                       その上に当たり判定を重ねて，指した点の語を出す
#
# **HTML は SVG を作り直さない。同じ SVG を中に入れる。** 別に描き直すと
# 図が2種類できて，どちらが正かが分からなくなる。注記（bursty な語の
# ラベル）も SVG の中にあるのでそのまま残る。
#
# 外部の JS ライブラリは使わない。CDN が塞がれたマシンでも開けるようにする。
INTERACTIVE_CSS = """
:root { --ink:#1f1e1b; --ink2:#5a5a55; --line:#d8d7d0; --surface:#ffffff;
        --wash:#f7f7f4; --accent:#184f95; }
* { box-sizing:border-box; }
body { margin:0; padding:24px 16px 48px; background:var(--wash);
       color:var(--ink); font-family:"Hiragino Sans","Noto Sans JP",
       "Yu Gothic",system-ui,sans-serif; line-height:1.6; }
.wrap { max-width:1100px; margin:0 auto; }
h1 { font-size:1.15rem; margin:0 0 .2em; font-weight:650; }
.sub { color:var(--ink2); font-size:.86rem; margin:0 0 1.1em; }
.card { background:var(--surface); border:1px solid var(--line);
        border-radius:10px; padding:14px; }
.figbox { position:relative; }
.figbox svg { width:100%; height:auto; display:block; }
#hit { position:absolute; inset:0; cursor:crosshair; }
#ring { position:absolute; width:22px; height:22px; margin:-11px 0 0 -11px;
        border:2px solid var(--accent); border-radius:50%;
        pointer-events:none; opacity:0; transition:opacity .08s; }
#tip { position:absolute; z-index:5; min-width:190px; max-width:290px;
       background:var(--surface); border:1px solid var(--line);
       border-radius:8px; box-shadow:0 6px 20px rgba(0,0,0,.13);
       padding:9px 11px; font-size:.8rem; pointer-events:none; opacity:0;
       transition:opacity .08s; }
#tip .term { font-size:1.05rem; font-weight:650; letter-spacing:.02em;
             margin-bottom:.35em; word-break:break-all; }
#tip dl { display:grid; grid-template-columns:auto 1fr; gap:1px 10px;
          margin:0; }
#tip dt { color:var(--ink2); font-size:.74rem; white-space:nowrap; }
#tip dd { margin:0; text-align:right; font-variant-numeric:tabular-nums;
          font-weight:600; }
.bar { display:flex; gap:10px; align-items:center; flex-wrap:wrap;
       margin:14px 0 0; font-size:.82rem; color:var(--ink2); }
.bar input { font:inherit; padding:5px 9px; border:1px solid var(--line);
             border-radius:6px; min-width:190px; background:var(--surface); }
.bar a { color:var(--accent); }
table { border-collapse:collapse; width:100%; font-size:.78rem;
        margin-top:10px; }
th,td { padding:4px 8px; border-bottom:1px solid #ecebe6; text-align:left;
        white-space:nowrap; }
th { background:var(--wash); position:sticky; top:0; font-weight:650; }
td.num { text-align:right; font-variant-numeric:tabular-nums; }
tbody tr:hover td { background:var(--wash); }
tbody tr.on td { background:#eaf1fb; }
.scroll { max-height:340px; overflow:auto; border:1px solid var(--line);
          border-radius:8px; margin-top:10px; }
.hint { font-size:.78rem; color:var(--ink2); margin:.6em 0 0; }
#links { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#links line { stroke:var(--accent); stroke-width:1.1; opacity:.55; }
#links circle { fill:none; stroke:var(--accent); stroke-width:1.4; opacity:.8; }
#marks { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#marks circle { fill:none; stroke:#d55e00; stroke-width:1.6; opacity:.9; }
#tip .notes { margin:.45em 0 0; font-size:.76rem; color:var(--ink);
              border-top:1px solid var(--line); padding-top:.4em;
              line-height:1.5; word-break:break-all; }
#tip .notes b { color:var(--ink2); font-weight:600; }
.prov { font-size:.72rem; color:var(--ink2); margin:.9em 0 0;
        border-top:1px solid var(--line); padding-top:.6em;
        font-variant-numeric:tabular-nums; }
.danger { background:#fdf0ea; border:1px solid #e8a37c; border-radius:8px;
          padding:9px 12px; font-size:.85rem; color:#8a3b10;
          margin:0 0 12px; }
"""

INTERACTIVE_JS = r"""
// 点は data-* ではなく JSON で渡す。語はコーパス由来の任意の文字列なので，
// **HTML に文字列連結で差し込まない**（textContent で入れる）。
// 見出し（keys・nhead）は全点で同じなら1回だけ入っている。点が1万個ある
// 図では，これで HTML が 1 MB 以上軽くなる。古い形（配列だけ）も読む。
const RAW = JSON.parse(document.getElementById('pts-data').textContent);
const PTS = Array.isArray(RAW) ? RAW : RAW.pts;
const KEYS = (RAW && RAW.keys) || [];
const NHEAD = (RAW && RAW.nhead) || '';
const LINKNOTES = !!(RAW && RAW.linknotes);
function pairsOf(p) {
  if (p.fields) return p.fields;
  if (p.v) return p.v.map((x, i) => [KEYS[i] || '', x]);
  return [];
}
function notesOf(p) {
  if (p.notes) return p.notes;
  if (p.n) return [NHEAD, p.n];
  // 本文が無く linknotes が立っているときは，線で結ぶ先の語を並べる
  if (LINKNOTES && p.links && p.links.length) {
    return [NHEAD, p.links.map(j => (PTS[j] || {}).term || '').join(' ')];
  }
  return null;
}
const box = document.getElementById('hit');
const tip = document.getElementById('tip');
const ring = document.getElementById('ring');
const rows = Array.from(document.querySelectorAll('tbody tr'));
const links = document.getElementById('links');
const marks = document.getElementById('marks');

// **最も近い点を拾う。** 点の直径は数ピクセルしかないので，
// 「真上に置く」ことを要求すると誰も当てられない（dataviz の規則）。
// カーソルに最も近い点を選び，遠すぎるときだけ何も出さない。
function nearest(px, py, w, h) {
  let best = null, bd = 1e9;
  for (const p of PTS) {
    const dx = p.x * w - px, dy = p.y * h - py;
    const d = dx * dx + dy * dy;
    if (d < bd) { bd = d; best = p; }
  }
  return Math.sqrt(bd) <= 34 ? best : null;   // 34px より遠ければ出さない
}

function fill(p) {
  tip.textContent = '';
  const h = document.createElement('div');
  h.className = 'term';
  h.textContent = p.term;                     // ← 連結しない
  tip.appendChild(h);
  const dl = document.createElement('dl');
  for (const [k, v] of pairsOf(p)) {
    const dt = document.createElement('dt'); dt.textContent = k;
    const dd = document.createElement('dd'); dd.textContent = v;
    dl.appendChild(dt); dl.appendChild(dd);
  }
  tip.appendChild(dl);
  const nt = notesOf(p);
  if (nt) {                                   // 近傍語など，横に長い情報
    const n = document.createElement('p');
    n.className = 'notes';
    const b = document.createElement('b');
    b.textContent = nt[0] + ' ';
    n.appendChild(b);
    n.appendChild(document.createTextNode(nt[1]));
    tip.appendChild(n);
  }
}

// **原空間での近傍を線で結ぶ。** 画面の近さは射影の結果にすぎない。
// 線が遠くへ伸びるなら，その点の近傍関係は2次元に収まっていない。
// これを見せるのが，この図でいちばん大事なところである。
function drawLinks(p, w, h) {
  if (!links) return;
  while (links.firstChild) links.removeChild(links.firstChild);
  if (!p.links || !p.links.length) return;
  const NS = 'http://www.w3.org/2000/svg';
  for (const j of p.links) {
    const q = PTS[j];
    if (!q) continue;
    const ln = document.createElementNS(NS, 'line');
    ln.setAttribute('x1', p.x * w); ln.setAttribute('y1', p.y * h);
    ln.setAttribute('x2', q.x * w); ln.setAttribute('y2', q.y * h);
    links.appendChild(ln);
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', q.x * w); c.setAttribute('cy', q.y * h);
    c.setAttribute('r', 5);
    links.appendChild(c);
  }
}

let cur = null;
function show(p, px, py) {
  const w = box.clientWidth, h = box.clientHeight;
  if (p !== cur) { fill(p); drawLinks(p, w, h); cur = p; }
  ring.style.left = (p.x * w) + 'px';
  ring.style.top = (p.y * h) + 'px';
  ring.style.opacity = 1;
  tip.style.opacity = 1;
  // はみ出さないように寄せる
  const tw = tip.offsetWidth, th = tip.offsetHeight;
  let lx = px + 16, ly = py + 14;
  if (lx + tw > w) lx = px - tw - 16;
  if (ly + th > h) ly = py - th - 14;
  tip.style.left = Math.max(0, lx) + 'px';
  tip.style.top = Math.max(0, ly) + 'px';
  rows.forEach(r => r.classList.toggle('on', r.dataset.i === String(p.r)));
}
function hide() {
  tip.style.opacity = 0; ring.style.opacity = 0; cur = null;
  if (links) while (links.firstChild) links.removeChild(links.firstChild);
  rows.forEach(r => r.classList.remove('on'));
}

box.addEventListener('pointermove', e => {
  const r = box.getBoundingClientRect();
  const p = nearest(e.clientX - r.left, e.clientY - r.top, r.width, r.height);
  if (p) show(p, e.clientX - r.left, e.clientY - r.top); else hide();
});
box.addEventListener('pointerleave', hide);

// 表の行にカーソルを乗せても，図の上の点が光る（逆引き）。
// **カーソルが使えない人にも同じ情報が届くように**，表を必ず添える
// （点が数千を超える図だけは表を絞る。絞ったことは図の下に明記する）。
rows.forEach(r => {
  r.addEventListener('mouseenter', () => {
    // 表の行は論理点。図の上では**先頭の面**の点を光らせる
    const p = PTS[Number(r.dataset.i)];
    if (!p) return;
    const w = box.clientWidth, h = box.clientHeight;
    show(p, p.x * w, p.y * h);
  });
  r.addEventListener('mouseleave', hide);
});

// 絞り込み。語・作品・時代のどれでも当たる
const q = document.getElementById('q');
if (q) q.addEventListener('input', () => {
  const s = q.value.trim();
  let n = 0;
  rows.forEach(r => {
    const hit = !s || r.textContent.includes(s);
    r.style.display = hit ? '' : 'none';
    if (hit) n++;
  });
  // 表を絞った図では，**表に無い語も図の上では当たる**。
  // 表の件数だけを出すと「無い」と誤解されるので両方を出す。
  const nlog = Number(document.body.dataset.nlog || rows.length);
  let extra = '';
  if (s && rows.length < nlog) {
    const seen = new Set();
    for (const p of PTS) if (p.term.includes(s)) seen.add(p.r);
    extra = '（図の上 ' + seen.size + ' 件）';
  }
  document.getElementById('count').textContent = n + ' 件' + extra;
  // **図の上にもマーカーを付ける。** 表だけ絞っても「どこにあるか」は分からない。
  if (!marks) return;
  while (marks.firstChild) marks.removeChild(marks.firstChild);
  if (!s) return;
  const NS = 'http://www.w3.org/2000/svg';
  const w = box.clientWidth, h = box.clientHeight;
  let drawn = 0;
  for (const p of PTS) {
    if (!p.term.includes(s)) continue;
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', p.x * w); c.setAttribute('cy', p.y * h);
    c.setAttribute('r', 7);
    marks.appendChild(c);
    if (++drawn > 400) break;          // マーカーが多すぎると図が読めない
  }
});
"""


def save_interactive(fig, ax, stem, xs, ys, tips, out=None, title='',
                     note='', table_cols=None, source=None, id_col='語',
                     hint='', table_idx=None, coords=None):
    """SVG を保存し，**同じ SVG を埋め込んだ対話的な HTML** も書く。

    ``xs`` ``ys`` はデータ座標，``tips`` は点ごとの情報
    （``{'term': 語, 'fields': [(見出し, 値), …]}`` の並び）。
    3つの長さは一致していなければならない。ずれたまま描くと，
    **指した点と出る語が食い違う**（注記の添字ずれと同じ事故）。

    位置は「図全体に対する割合」で書き出す。SVG を ``width:100%`` で
    伸縮させても割合は変わらないので，どんな幅でも点と当たり判定が
    合う。座標は matplotlib の変換を通して得るので，**図と HTML で
    座標の計算が二重にならない**。

    ``source`` に入力ファイルのパスを渡すこと。**どの表から描いた図かを
    HTML の末尾に刻む。** これが無いと，試験用の作りかけのデータから
    描いた図と，本番のデータから描いた図が見分けられない。
    入力がプロジェクトの外（``/tmp`` など）にあるときは
    「試験用」と赤字で出し，配布してはいけないことを図自身に言わせる。

    ``id_col`` は表の第1列の見出し（既定「語」。作品を点にする図では
    「作品」などに変える）。

    ``ax`` には**面の並び**も渡せる（``[axes[0], axes[1]]``）。同じ点を
    別の色分けで2面に描いた図では，どちらの面を指しても同じ情報が出る。
    表の行は点ごとに1行だけ作る（面の数だけ重複させない）。

    ``table_idx`` は**表に載せる点の添字**（既定は全点）。点が数千を超える
    図では表を全件出すと HTML が数 MB になり，読む側にも役に立たない。
    そのときは載せる点を選ぶ。**ただし図の当たり判定と検索は全点に効く**
    ので，表に無い語も指せるし検索で図にマーカーが付く。表を絞ったときは，
    何件のうち何件を載せたかを HTML に明記する（黙って捨てないこと）。

    ``coords`` は**面ごとの座標**（``[(x1, y1), (x2, y2)]``）。同じ点を
    **違う座標系**で2面に描いた図（主成分分析と UMAP の比較など）で使う。
    渡さなければ全部の面で ``xs`` ``ys`` を使う。
    ⚠ 面ごとに座標が違うのに ``coords`` を渡さないと，2面めの当たり判定が
    1面めの座標で置かれる。**図は出るが，指した点と出る語が食い違う。**
    """
    import json as _json
    if not (len(xs) == len(ys) == len(tips)):
        raise ValueError(
            f'save_interactive: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'情報={len(tips)}）。座標と情報を同じ添字で絞り込むこと。')

    svg_path = save_fig(fig, stem, out=out)
    outdir = svg_path.parent

    # ---- 埋め込む SVG は**切り取らずに**保存する ------------------------
    # save_fig は bbox_inches='tight' で余白を詰めるため，図全体に対する
    # 割合と，ファイルの座標系がずれる。埋め込み用は詰めずに出す。
    import io
    buf = io.StringIO()
    fig.savefig(buf, format='svg', dpi=RASTER_DPI)
    svg = buf.getvalue()
    svg = svg[svg.index('<svg'):]          # XML 宣言と DOCTYPE を落とす

    # ---- 点の位置を図全体に対する割合で得る -----------------------------
    axes_list = list(ax) if isinstance(ax, (list, tuple, np.ndarray)) else [ax]
    W, H = fig.bbox.width, fig.bbox.height
    n_pts = len(tips)

    # **見出しは点ごとに書かない。** 点が1万個ある図では，
    # 「品詞」「頻度」…という見出しを1万回繰り返すだけで HTML が
    # 1 MB 以上ふくらむ。全点で見出しが同じなら1回だけ書き，
    # 値の並びだけを点に持たせる（JS 側で組み直す）。
    keys = [str(k) for k, _ in tips[0].get('fields', [])] if tips else []
    same_keys = bool(keys) and all(
        [str(k) for k, _ in t.get('fields', [])] == keys for t in tips)
    nheads = {str(t['notes'][0]) for t in tips if t.get('notes')}
    nhead = next(iter(nheads)) if len(nheads) == 1 else ''
    # 本文を渡さず ``notes=(見出し, None)`` としたときは，
    # ``links`` の先の語を JS 側で並べる
    linknotes = bool(nhead) and any(
        t.get('notes') and t['notes'][1] is None and t.get('links')
        for t in tips)

    pts = []
    if coords is not None and len(coords) != len(axes_list):
        raise ValueError(
            f'save_interactive: coords の数が面の数と違う'
            f'（面 {len(axes_list)} / coords {len(coords)}）')
    for k, axk in enumerate(axes_list):
      xk, yk = (coords[k] if coords is not None else (xs, ys))
      if not (len(xk) == len(yk) == n_pts):
          raise ValueError(
              f'save_interactive: 面 {k} の座標の数が情報の数と違う'
              f'（x={len(xk)}, y={len(yk)}, 情報={n_pts}）')
      pxy = axk.transData.transform(np.column_stack([np.asarray(xk, float),
                                                     np.asarray(yk, float)]))
      for j, (t, (px, py)) in enumerate(zip(tips, pxy)):
        i = k * n_pts + j
        # **変数名に注意。** ここを d と書くと，上で取った出力先 d
        # （svg_path.parent）を上書きして，最後に d / '....html' が
        # 「dict ÷ str」になる。名前は使い回さない。
        # 添字 i は JS では使わない（行は r で引く）。点が1万個ある図では
        # 使わない値も 100 KB 単位で効くので書かない。
        rec = {'r': j, 'term': str(t.get('term', '')),
               'x': round(float(px) / W, 6),
               'y': round(1 - float(py) / H, 6)}        # SVG は上が 0
        if same_keys:
            rec['v'] = [str(b) for _, b in t.get('fields', [])]
        else:
            rec['fields'] = [[str(a), str(b)] for a, b in t.get('fields', [])]
        if t.get('notes'):
            # ('見出し', '本文') の2つ組。横に長い情報（近傍語など）。
            # 本文を None にすると，**線で結ぶ先の語を JS が並べる**
            # （同じ語の列を点ごとに書かずに済む。1万点で 1 MB 近く効く）
            if t['notes'][1] is None:
                pass
            elif nhead:
                rec['n'] = str(t['notes'][1])
            else:
                rec['notes'] = [str(t['notes'][0]), str(t['notes'][1])]
        if t.get('links'):
            # 原空間での近傍の添字。**同じ面の中で**線を結ぶ
            rec['links'] = [k * n_pts + int(q) for q in t['links']]
        pts.append(rec)

    cols = table_cols or keys
    head = f'<tr><th>{_esc(id_col)}</th>' + ''.join(
        f'<th>{_esc(c)}</th>' for c in cols) + '</tr>'
    # 数字の列だけ右寄せにする。時代名や作品 ID を右寄せにすると読みにくい。
    def _numish(v):
        t = str(v).strip().replace('%', '').replace(',', '')
        t = t.lstrip('+-')
        return bool(t) and t.replace('.', '', 1).isdigit()

    # 表に載せる点。**面の数だけ重複させない**（論理点1つに1行）
    if table_idx is None:
        order = list(range(n_pts))
    else:
        seen, order = set(), []
        for i in [int(q) for q in table_idx]:   # 重複を除きつつ順序は保つ
            if 0 <= i < n_pts and i not in seen:
                seen.add(i); order.append(i)

    # 表は **tips から作る**（点の JSON は見出しを省いてあるので）
    body = []
    for j in order:
        fv = {str(a): str(b) for a, b in tips[j].get('fields', [])}
        tds = ''
        for c in cols:
            v = fv.get(c, '')
            cls = ' class="num"' if _numish(v) else ''
            tds += f'<td{cls}>{_esc(v)}</td>'
        body.append(f'<tr data-i="{j}">'
                    f'<td>{_esc(tips[j].get("term", ""))}</td>{tds}</tr>')

    # ---- 由来を図自身に刻む -------------------------------------------
    # **どの表から描いた図かが分からないと，試験用のデータで描いた図が
    # 本物として配られかねない。**
    import datetime as _dt
    stamp = _dt.datetime.now().astimezone().strftime('%Y-%m-%d %H:%M')
    src = Path(source) if source else None
    prov = f'点 {len(pts)} 個／作図 {stamp}'
    warn = ''
    if src is not None:
        try:
            mt = _dt.datetime.fromtimestamp(src.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        except OSError:
            mt = '不明'
        prov = f'入力 {src.name}（更新 {mt}）／' + prov
        # プロジェクトの外（/tmp など）から描いた図は試験用である
        try:
            outside = not str(src.resolve()).startswith(str(ROOT.resolve()))
        except Exception:                               # noqa: BLE001
            outside = True
        if outside or '/tmp/' in str(src):
            warn = ('<p class="danger">⚠ <b>試験用の入力から作った図である。'
                    f'配布してはいけない。</b>（入力 {_esc(str(src))}）</p>')
            prov = f'入力 {_esc(str(src))}／' + f'点 {len(pts)} 個／作図 {stamp}'

    html = f"""<!DOCTYPE html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{_esc(title or stem)}</title>
<style>{INTERACTIVE_CSS}</style></head>
<body data-nlog="{n_pts}" data-ntable="{len(body)}"><div class="wrap">
<h1>{_esc(title or stem)}</h1>
<p class="sub">{_rich(note)}</p>
{warn}
<div class="card">
  <div class="figbox">
    {svg}
    <svg id="links"></svg><svg id="marks"></svg>
    <div id="hit"></div><div id="ring"></div><div id="tip"></div>
  </div>
  <p class="hint">{_rich(hint or '点にカーソルを近づけると語が出る（最も近い点を拾うので，真上に置かなくてよい）。図の中の注記は静止版と同じものである。')}</p>
  <div class="bar">
    <input id="q" type="search" placeholder="語・作品・時代で絞り込む">
    <span id="count">{len(body)} 件</span>
    <span>·</span>
    {(f'<span>表は {len(body)} 件（図の点は {n_pts} 件。'
      '表に無い語も図の上で指せる。検索は図のマーカーにも効く）</span><span>·</span>')
     if len(body) < n_pts else ''}
    <a href="{_esc(svg_path.name)}" download>SVG を保存</a>
    <span>（この HTML の中の図はその SVG そのもの）</span>
  </div>
  <div class="scroll"><table><thead>{head}</thead>
    <tbody>{''.join(body)}</tbody></table></div>
  <p class="prov">{prov}</p>
</div>
<script type="application/json" id="pts-data">{_json.dumps(
    {'keys': keys if same_keys else [], 'nhead': nhead,
     'linknotes': linknotes, 'pts': pts},
    ensure_ascii=False, separators=(',', ':'))}</script>
<script>{INTERACTIVE_JS}</script>
</div></body></html>
"""
    path = outdir / f'{stem}.html'
    path.write_text(html, encoding='utf-8')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB・対話版／'
          f'点 {len(pts)} 個)')
    return svg_path, path


def _esc(s):
    """HTML の特殊文字を落とす。**語はコーパス由来なので必ず通す。**"""
    return (str(s).replace('&', '&amp;').replace('<', '&lt;')
            .replace('>', '&gt;').replace('"', '&quot;'))


def _rich(s):
    """注記の ``**…**`` だけを太字にする。

    説明文をノートブックと同じ書き方（Markdown 風）で書けるようにする。
    **先に必ず _esc を通す**ので，タグを書き込まれる余地は無い。
    ``**`` のままだと HTML では記号がそのまま出て読みにくい。
    """
    import re as _re
    return _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', _esc(s))


PALETTE = ['#0072B2', '#E69F00', '#009E73', '#CC79A7',
           '#56B4E9', '#D55E00', '#F0E442', '#666666']

# --- 順序のあるものを色分けするための1色相のランプ -----------------------------
# **時代・年次・段階のように順序のあるものを，上の8色で色分けしてはいけない。**
# 明治中期が青で明治後期が黄なら，隣り合う時代が隣り合う色にならず，
# 「時代が下るとどちらへ動くか」という肝心のことが読めなくなる。
# 1色相の濃淡にすれば，近いもの同士が近い色になり，勾配がそのまま見える。
# 散布図の点は白地の上に置くので，いちばん明るい段は 100 ではなく
# 250（背景との対比 2:1）から始める。100 は面を色分けするとき（ヒートマップ）用。
SEQ_BLUE_STEPS = ['#86b6ef', '#5598e7', '#3987e5',
                  '#256abf', '#184f95', '#0d366b']
SEQ_BLUE = LinearSegmentedColormap.from_list('jlit_blue', SEQ_BLUE_STEPS)

# 離散の順序（4区分など）を色分けするときはこちら。隣の段と明度差が十分あり，
# いちばん明るい段も背景から浮く（対比 2:1 以上）ことを確かめてある。
SEQ_BLUE_5 = ['#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b']

# 大分類の2色。散布図ではどの2点も隣り合いうるので**全ペアが
# 見分けられる必要**があり，使える色数は多くない。2色に絞って，
# 下位の区別はマーカーの形に持たせる。
GENRE_C = {'Fiction': '#2a78d6', 'Nonfiction': '#eb6834'}

# --- 初出年の5段 ---------------------------------------------------------
# 切れ目は period と同じ 1900／1912／1926／1945。
# **6段にはできない。** 1色相の濃淡で順序を見せるには，隣り合う段の明度差が
# 0.06 以上要る。この青系ランプは 250→700 で明度差にして 0.30 ほどしか幅が
# 無いので，段を6つ取るとどこかが 0.05 台に落ち，隣が見分けられなくなる。
# そこで作品数3点の明治前期（〜1886）を明治中期にまとめて5段とする。
YEAR_EDGES = [1900, 1912, 1926, 1945]
YEAR_LABELS = ['〜1899 明治前・中期', '1900-1911 明治後期', '1912-1925 大正',
               '1926-1944 昭和戦前', '1945- 昭和戦後']


def year_bands(years):
    """初出年を5段に畳み，``(段番号, ラベル, 色)`` を返す。

    段番号は 0〜4。**初出年が読めないものは -1** にする。0 に落とすと
    年の分からない作品が全部いちばん古い段に入り，通時の議論が崩れる。

    時代で色分けする図はすべてこれを通すこと。同じ色が全ステップで同じ時代を
    指すようになり，Step 1 の図と Step 7 の図を並べて読める。
    """
    y = pd.to_numeric(pd.Series(list(years)), errors='coerce')
    code = np.full(len(y), -1, dtype=int)
    ok = y.notna().values
    if ok.any():
        code[ok] = np.digitize(y[ok].values, YEAR_EDGES)
    return code, YEAR_LABELS, SEQ_BLUE_5


def run_script(script, *args, tail=4000):
    """scripts/ のスクリプトを実行し，標準出力・標準エラー・終了コードを必ず表示する。

    print(r.stdout or r.stderr) では，標準出力が空でないときに
    エラーの内容が隠れてしまう。学習用には両方見えるほうがよい。
    """
    cmd = [sys.executable, str(ROOT / 'scripts' / script)] + [str(a) for a in args]
    print('$ python', ' '.join(cmd[1:]))
    r = subprocess.run(cmd, capture_output=True, text=True)

    def _tail(s):
        # 末尾 tail 文字だけを出す。行の途中で切らないよう，切ったときは
        # 次の改行から始め，前を省いたことを明示する
        if len(s) <= tail:
            return s
        s = s[-tail:]
        return '（…前略）\n' + s[s.find('\n') + 1:]

    if r.stdout:
        print(_tail(r.stdout))
    if r.stderr.strip():
        # 標準エラーには**エラー以外**も出る。MALLET は学習の進み具合
        # （<10> LL/token: …）と途中のトピック上位語をここに書く。
        # 判断は [exit 0] かどうかで行う
        print('--- stderr（進行ログを含む。エラーとは限らない）---')
        print(_tail(r.stderr))
    print(f'[exit {r.returncode}]' + ('' if r.returncode == 0 else '  ← 0 でなければ失敗'))
    return r


# 使うメタデータ。増補分を含む v3 があればそちらを優先する。
# v2 は v1 の 64 点しか無いので，増補後のコーパスで v2 を使うと
# 突合が外れて period も genre も空になる（Step 3 で v3 を作る）。
# Step 3 で自分が作った v3 は *_local.csv に書かれる（配布版は上書きしない。
# 上書きすると git pull のたびに衝突する）。自分の版 > 配布版 v3 > v2 の順。
for _m in ('corpus_metadata_v3_local.csv', 'corpus_metadata_v3.csv',
           'corpus_metadata_v2.csv'):
    META = ROOT / 'metadata' / _m
    if META.exists():
        break

# 図・表の書き出し先は自分の作業フォルダ my_work/results/。my_work/ は
# コースのリポジトリの外扱い（.gitignore）で，自分の GitHub にバックアップを取る。
OUT = ROOT / 'my_work' / 'results'
OUT.mkdir(parents=True, exist_ok=True)
print('OUT  =', OUT)


## 1. Procrustes アラインメントを手で実装する

まず小さな例で，回転してもコサインが変わらないことを確かめる。

In [ ]:
rng = np.random.default_rng(0)
A = rng.normal(size=(200, 50))
Q,_ = np.linalg.qr(rng.normal(size=(50,50)))       # ランダムな直交行列
B = A @ Q                                          # A を回転しただけ

def cos(a,b): return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
print('同じ2語の内部的なコサイン（A空間）:', round(cos(A[0],A[1]),4))
print('同じ2語の内部的なコサイン（B空間）:', round(cos(B[0],B[1]),4), '← 不変')
print('A と B の「同じ語」のコサイン          :', round(cos(A[0],B[0]),4), '← 無意味')

def procrustes(base, other):
    U,_,Vt = np.linalg.svd(other.T @ base)
    return U @ Vt
R = procrustes(A, B)
print('アラインメント後の「同じ語」のコサイン :', round(cos(A[0], (B@R)[0]),4), '← 回復')

## 2. 時代スライスの設計

スライスの切り方は**結果を決める**。本コーパスの制約を思い出す。

| 時代区分 | 語数 | 判断 |
|---|---:|---|
| 明治前期（〜1886） | 54,035 | 単独では学習不可 |
| 明治中期（1887–1899） | 222,610 | 単独では不安定 |
| 明治後期（1900–1911） | 682,188 | ぎりぎり |
| 大正（1912–1925） | 1,168,853 | 可 |
| 昭和戦前（1926–1944） | 2,556,493 | 可 |
| 昭和戦後（1945–） | 1,366,057 | 可 |

**明治期3区分を合わせても 96万語**。そのままでは 6 スライスに切れない。
現実的な選択肢は

- (a) 明治（〜1911）／大正（1912–25）／昭和戦前／昭和戦後 の **4スライス**
- (b) 戦前（〜1944）／戦後 の **2スライス**（最も頑健）
- (c) 増補を待って 6 スライス

本授業では (a) を既定とし，(b) で結果を確認する。

In [ ]:
DS = ROOT/'data'/'datasets'
ci = pd.read_csv(DS/'chunks_index.csv')

# period が空のチャンクを先に始末する。**ここが通時分析の分かれ目である。**
# 06 の突合が外れると period は NaN になる。NaN を「その他」に落とす
# 書き方（`return 'D_昭和戦後'` のような最後の return）だと，
# **素性の知れないチャンクが昭和戦後に混ざったまま**通時変化の図ができる。
# 時代が分からないものは，時代の分析から外すのが正しい。
bad = ci.period.isna() | (ci.period.astype(str).str.strip() == '')
if bad.any():
    LAB = work_labels()
    works = sorted({str(w) for w in ci.loc[bad, 'work_stem']})
    print(f'[warn] **period が空のチャンクが {int(bad.sum()):,} / {len(ci):,} ある**')
    print(f'       作品数にして {len(works)} 点。時代スライスから外す。')
    # 作品名だけを流すのではなく，**何チャンク落ちるか**まで出す。
    # 落ちる量が分からないと，無視してよい漏れかどうかが判断できない。
    cnt = ci.loc[bad, 'work_stem'].astype(str).value_counts()
    show(pd.DataFrame({'作品': [LAB.get(w, w) for w in cnt.index],
                       'work_stem': cnt.index,
                       '落ちるチャンク数': cnt.values}),
         caption='時代スライスから外れる作品（period が空）')
    print('       原因は 06_build_datasets.py の突合漏れである。')
    print('       Step 3 で 00_extend_metadata.py → 06 を走らせ直し，')
    print('       meta_unmatched.csv が空になってから戻ってくること。')

def coarse(p):
    p = str(p)
    if p.startswith(('1_','2_','3_')): return 'A_明治(〜1911)'
    if p.startswith('4_'):             return 'B_大正(1912-25)'
    if p.startswith('5_'):             return 'C_昭和戦前(1926-44)'
    if p.startswith('6_'):             return 'D_昭和戦後(1945-)'
    # ここに来るのは period の綴りが想定外のとき。黙って昭和戦後に
    # 入れてはいけない。NaN にして後段の dropna で落とす。
    return None

ci['slice4'] = ci.period.map(coarse)
ci['slice2'] = np.where(
    ci.period.astype(str).str.startswith(('1_','2_','3_','4_','5_')),
    'PRE_戦前', 'POST_戦後')

ci.loc[bad, 'slice2'] = None          # 時代不明は戦前/戦後の別も付けない

unknown = ci.slice4.isna()
if unknown.any() and not bad.all():
    odd = sorted({str(x) for x in ci.loc[unknown & ~bad, 'period']})
    if odd:
        print(f'[warn] period の綴りが想定外のチャンクがある: ' + '，'.join(odd[:5]))

# **索引そのものは削らずに書き戻す。** ここで `ci = ci[~bad]` としてから
# 保存すると，突合の外れたチャンクが索引から消え，あとから
# 「何件落ちたのか」を確かめられなくなる。列を足すだけにして，
# 分析には下の `sl` を使う。
ci.to_csv(DS/'chunks_index.csv', index=False, encoding='utf-8-sig')

sl = ci.dropna(subset=['slice4'])      # ← 以後のスライス学習はこれを使う
print(sl.slice4.value_counts().sort_index().to_string())
print()
print(sl.slice2.value_counts().to_string())
print(f'\nスライスに使うチャンク {len(sl):,} / {len(ci):,} 件'
      f'（作品 {sl.work_stem.nunique()} 点）')

### 統制すべき交絡（Dubossarsky et al. 2017 の警告）

彼らは，**語をランダムにシャッフルした「偽の時系列」でも
意味変化の法則（頻度が高い語ほど変化しない等）が再現されてしまう**ことを示した。
つまり，観測された「法則」の多くは word embedding の統計的性質の産物である。

したがって最低限，次を統制する。

1. **スライスごとの語数を揃える**（`--balance`）
2. **作家の偏りを抑える**（`--max-per-author`）
3. **乱数を変えて複数回**学習し，安定した変化だけを報告する
4. **コントロール実験**：時代ラベルをシャッフルして同じ手順を回し，
   本物の変化量が偽の変化量を上回るか確かめる

In [ ]:
W2V = OUT/'w2v_slice4'
# --runs は**シードを変えて何回学習するか**。既定は 10（config/pipeline.yaml）。
# 授業では時間の都合で 3 回にしてある。**報告には回数を必ず書くこと**
# （10 回で測り直すなら --runs を外すか 10 を渡す）。
# スライス別モデルを runs 回ぶん学習するので，時間はおよそ runs 倍かかる。
RUNS = 3
run_script('08_word2vec_diachronic.py', '--chunks', DS/'chunks',
           '--index', DS/'chunks_index.csv', '--out', W2V, '--slice', 'slice4',
           '--dim', 300, '--window', 3, '--min-count', 20, '--balance',
           '--runs', RUNS)
print(f'※ {RUNS} 回の平均で測った。semantic_change.csv の drift_sd が'
      'ばらつきである。**ばらつきの大きい語で意味変化を論じてはいけない。**')

In [ ]:
p = W2V/'semantic_change.csv'
if need(p, '08_word2vec_diachronic.py を先に走らせること'):
    sc = pd.read_csv(p)
    # **平均だけを見せない。** 試行間のばらつき（drift_sd）と変動係数
    # （drift_cv = sd/平均）を必ず並べる。変化量が大きくても
    # ばらつきが大きい語は「乱数で動いただけ」かもしれない。
    cols = {'term': '語', 'drift_first_last': '変化量（平均）',
            'drift_sd': 'ばらつき（SD）', 'drift_cv': '変動係数',
            'drift_min': '最小', 'drift_max': '最大',
            'max_step_at': '最大変化の時代', 'runs': '試行'}
    have = [c for c in cols if c in sc.columns]
    t = sc[have].head(25).rename(columns=cols)
    if '変動係数' in t:
        # ばらつきの大きい語に印を付ける（色だけに頼らず文字で示す）
        t['判定'] = np.where(pd.to_numeric(t['変動係数'],
                                          errors='coerce') > 0.25,
                             '⚠ 解釈に耐えない', '')
    show(t, caption=f'意味変化量の大きい語（上位25・'
                    f'{int(sc.runs.iloc[0]) if "runs" in sc else 1} 回の平均）',
         fmt={'変化量（平均）': '{:.4f}', 'ばらつき（SD）': '{:.4f}',
              '変動係数': '{:.3f}', '最小': '{:.4f}', '最大': '{:.4f}',
              '試行': '{:.0f}'})
    if 'drift_cv' in sc.columns:
        cv = pd.to_numeric(sc.drift_cv, errors='coerce')
        n_bad = int((cv.head(100) > 0.25).sum())
        print(f'上位100語のうち，試行間のばらつきが大きい語は {n_bad} 語。'
              '**この語で意味変化を論じてはいけない。**')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    axes[0].hist(sc.drift_first_last, bins=50, color=PALETTE[0], alpha=.85)
    axes[0].set_xlabel('明治→昭和戦後 のコサイン距離（平均）')
    axes[0].set_ylabel('語数'); axes[0].set_title('意味変化量の分布')
    if 'drift_sd' in sc.columns and sc.drift_sd.max() > 0:
        # **変化量とばらつきを同じ図で見る。** 右下（大きく・安定）の語が
        # 解釈に値する。対角線より上の語は，変化量よりも揺れのほうが大きい。
        axes[1].scatter(sc.drift_first_last, sc.drift_sd, s=8,
                        color=PALETTE[0], alpha=.45, rasterized=True)
        lim = float(max(sc.drift_first_last.max(), sc.drift_sd.max()))
        axes[1].plot([0, lim], [0, lim * 0.25], color=PALETTE[5], lw=1,
                     ls='--', label='変動係数 0.25')
        axes[1].set_xlabel('変化量（平均）'); axes[1].set_ylabel('ばらつき（SD）')
        axes[1].set_title('破線より上は「乱数で動いた」語')
        axes[1].legend(frameon=False, fontsize=9)
    else:
        axes[1].text(.5, .5, '1回しか学習していないので\nばらつきは測れない',
                     ha='center', va='center', fontsize=11)
        axes[1].set_axis_off()
    for a_ in axes:
        a_.spines[['top','right']].set_visible(False)
    fig.tight_layout(); save_fig(fig, 'Step6_drift_hist'); plt.show()

## 3. コントロール実験 — 「偽の時代」と比べる

時代ラベルをシャッフルして同じ手順を回す。**本物の変化量が
偽の変化量とほとんど変わらなければ，観測された変化は時代効果ではない。**

これは本授業で最も重要な手続きである。

In [ ]:
# 時代ラベルをシャッフルした対照条件
#
# **ラベルの付いた行だけを入れ替える。** 索引ぜんぶを permutation にかけると，
# 時代不明（slice4 が空）の行のぶんだけ空ラベルが混ざり込み，
# 本物の条件と対照条件でチャンク数が変わってしまう。比較にならない。
ci_shuf = sl.copy()
rs = np.random.default_rng(20260920)
ci_shuf['slice4'] = rs.permutation(ci_shuf['slice4'].values)
shuf_path = DS/'chunks_index_shuffled.csv'
ci_shuf.to_csv(shuf_path, index=False, encoding='utf-8-sig')
print(f'対照条件のチャンク {len(ci_shuf):,} 件（本物の条件と同数であること）')

W2V_S = OUT/'w2v_shuffled'
run_script('08_word2vec_diachronic.py', '--chunks', DS/'chunks',
           '--index', shuf_path, '--out', W2V_S, '--slice', 'slice4',
           '--dim', 300, '--window', 3, '--min-count', 20, '--balance',
           '--runs', RUNS, tail=1500)
# **回数も本物の条件と同じにする。** 片方だけ10回にすると，
# 平均の安定度が違うぶんだけ分布の幅が変わり，比較にならない。

In [ ]:
a = W2V/'semantic_change.csv'; b = W2V_S/'semantic_change.csv'
if a.exists() and b.exists():
    real = pd.read_csv(a); fake = pd.read_csv(b)
    fig, ax = plt.subplots(figsize=(8,5))
    ax.hist(real.drift_first_last, bins=50, alpha=.7,
            color=PALETTE[0], label='本物の時代ラベル', density=True)
    ax.hist(fake.drift_first_last, bins=50, alpha=.7,
            color=PALETTE[5], label='シャッフルした時代ラベル', density=True)
    ax.set_xlabel('コサイン距離'); ax.set_ylabel('密度')
    ax.set_title('意味変化量：本物 vs 対照条件')
    ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
    fig.tight_layout(); save_fig(fig, 'Step6_control'); plt.show()
    q = lambda s: {'語数': len(s), '中央値': s.median(),
                   '第1四分位': s.quantile(.25), '第3四分位': s.quantile(.75),
                   '最大': s.max()}
    show(pd.DataFrame([{'条件': '本物の時代ラベル', **q(real.drift_first_last)},
                       {'条件': 'シャッフルした対照', **q(fake.drift_first_last)}]),
         caption='意味変化量（最初のスライスと最後のスライスのコサイン距離）',
         fmt={'中央値':'{:.4f}','第1四分位':'{:.4f}',
              '第3四分位':'{:.4f}','最大':'{:.4f}'})
    d = real.drift_first_last.median() - fake.drift_first_last.median()
    print(f'中央値の差 {d:+.4f}。**2つの分布が重なるほど，'
          f'「時代差」の主張は弱くなる。**')

## 4. 近傍語の変遷を読む

数値の大小より，**近傍語がどう入れ替わったか**のほうが解釈に堪える。

In [ ]:
p = W2V/'probe_neighbours.csv'
if need(p, '08_word2vec_diachronic.py を先に走らせること'):
    nb = pd.read_csv(p)
    # 語ごとにブロックを作り，語名は先頭行だけに出す。
    # print で流すと，同じスライスの行が縦に揃わず比べにくい。
    rows = []
    for term in nb.term.unique()[:8]:
        g = nb[nb.term==term]
        for i,(_,r) in enumerate(g.iterrows()):
            rows.append({'語': term if i==0 else '',
                         'スライス': r['slice'], '近傍語': r.neighbours})
    show(pd.DataFrame(rows), caption='時代スライスごとの近傍語')
    print('**縦に読むのではなく，同じ語の行を横に見比べること。**'
          '入れ替わった語が意味変化の候補である。')

### 演習 — 自分の5語を追う

Step 5 で選んだ5語について
1. 変化量（drift）はどの程度か
2. 近傍語はどう入れ替わったか
3. 対照条件と比べて，その変化は意味があるか
4. **原文で確かめる。** KWIC で実際の用例を読む。数値だけで論じない。

In [ ]:
# 簡易 KWIC。数値で見つけた変化を，必ず原文で確認する。
PLAIN = ROOT/'data'/'plain'/'full'
meta = load_meta()
# キーは work_rows() に作らせる。自前で f'{person_id}_{work_id}' と書くと
# **0 埋めの違いで1件も一致せず**，period_prefix での絞り込みが黙って
# 効かなくなる（全作品が対象になる）。ラベルも出なくなる。
ROWS = work_rows(meta)
LAB  = work_labels(with_year=True)

miss = [f.stem for f in sorted(PLAIN.glob('*.txt')) if f.stem not in ROWS]
if miss:
    print(f'[warn] メタデータに無い本文 {len(miss)} 件: ' + '，'.join(miss[:5]))

def kwic(word, width=28, limit=12, period_prefix=None):
    """簡易 KWIC。**表にして出す。**

    左文脈を右寄せにすると，キーワードが縦に揃って並ぶ。
    print で流すと，全角の幅のせいで揃わない。
    """
    rows = []
    for f in sorted(PLAIN.glob('*.txt')):
        r = ROWS.get(f.stem)
        if period_prefix:
            if r is None:                      # 年が分からないものは外す
                continue
            if not str(r.get('year_first')).startswith(period_prefix):
                continue
        t = f.read_text(encoding='utf-8').replace('\n','　')
        lab = LAB.get(f.stem, f.stem)
        for i in range(len(t)):
            if t.startswith(word, i):
                rows.append({'作品': lab,
                             '左文脈': '…' + t[max(0,i-width):i],
                             '語': word,
                             '右文脈': t[i+len(word):i+len(word)+width] + '…'})
                if len(rows) >= limit:
                    break
        if len(rows) >= limit:
            break
    if not rows:
        print(f'「{word}」は見つからなかった'
              + (f'（{period_prefix} 台に限定）' if period_prefix else ''))
        return
    cap = f'KWIC：「{word}」' + (f'（{period_prefix} 台）' if period_prefix else '')
    show(pd.DataFrame(rows), caption=f'{cap}　{len(rows)} 例',
         align={'左文脈': 'right', '語': 'center', '右文脈': 'left'})

kwic('自由')

## 5. このステップの課題

次の設問への答えを，テンプレート `my_work/results/Step6_report.md` に書いて提出する（**全体で600–1000字程度**。図表と「再現のための情報」は字数に含めない）。

- **提出先**：Zulip（dh-uosaka.zulipchat.com）の非公開チャネル **2026年度テクスト分析論B** ＞ トピック **Step 6**
- テンプレートの中身をメッセージに貼り付け，図（SVG）・表（CSV）は**同じメッセージに添付**する（1人1通）
- 図は番号で言及し（図1），**図を見なくても論旨が追えるように**書く（SVG は Zulip で表示されないことがある）
- 再提出は元の投稿を直さず，同じトピックに新しく投稿する（手順書 §5.3）

1. 4スライスと2スライスの両方で実行し，結果の頑健性を比較すること。
2. **対照条件（シャッフル）の図を必ず添付**し，そこから言えることを述べること。
3. 自分の5語について，drift・近傍語・KWIC の3点セットで報告すること。
4. 「この語は意味が変化した」と言えるための条件を，自分の言葉で3つ挙げること。

### このステップの到達点（次へ進む条件）

- 時代スライスごとのモデルを学習し，Procrustes でアラインメントできた
- **シャッフル対照**を実行し，本物の変化量と比較した図がある
- 変化が大きいと出た語について，KWIC で原文を確認した
- スライス間の語数の偏りを `--balance` で吸収した
